# Geometric Hallucination Detection with SGI on HaluEval

An independent reproduction attempt of the context-based Semantic Grounding Index (SGI) discussed in [Javier Marín, *A Geometric Taxonomy of Hallucinations in LLMs*](https://arxiv.org/html/2602.13224v1). This notebook focuses on HaluEval only. It does not implement the paper's context-free Directional Grounding Index.

For normalized embeddings, angular distance is `theta(a, b) = arccos(clip(a · b, -1, 1))`, and `SGI = theta(response, question) / theta(response, context)`. Higher scores suggest stronger context grounding; this is a geometric signal, not a factuality guarantee.

## Recorded progress before cleanup

The original saved outputs reported 64,507 normalized response rows: 20,000 each for QA, dialogue and summarization, plus 4,507 general-query rows. Filtering yielded 60,000 complete triplets. The embedding cell reported a cache hit for `all-mpnet-base-v2`. The external Drive cache is not included in this repository and has not been independently verified here. No SGI values, AUROC or effect sizes were saved in the original notebook.

The scoring and evaluation sections below were added during cleanup. A subsequent Colab run on November 4, 2026 produced results now recorded in README.md and results/HaluEval/. This local notebook does not contain that run's execution outputs. All execution outputs were cleared to avoid presenting historical outputs as runs of the revised code.


## 1. Environment

Run in Google Colab with a GPU and Google Drive access. CPU execution is supported but slower. Downloads require internet access. Dependencies are installed without version pins; record the generated environment file when comparing runs.


In [ ]:
%pip -q install transformers torch numpy pandas pyarrow scikit-learn tqdm

import gc
import hashlib
import importlib.metadata
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from google.colab import drive
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "HaluEval"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "by_dataset" / "HaluEval"
RESULTS_DIR = PROJECT_ROOT / "results" / "HaluEval"
for path in [RAW_DIR.parent, PROCESSED_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
versions = {name: importlib.metadata.version(name) for name in
            ["transformers", "torch", "numpy", "pandas", "pyarrow", "scikit-learn"]}
(RESULTS_DIR / "environment.json").write_text(json.dumps(versions, indent=2))
print("Device:", DEVICE)


## 2. Download HaluEval

Clone the official repository once and record its revision. Existing checkouts are reused without pulling updates.


In [ ]:
if not (RAW_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/RUCAIBox/HaluEval.git", str(RAW_DIR)], check=True)
DATA_REVISION = subprocess.check_output(
    ["git", "-C", str(RAW_DIR), "rev-parse", "HEAD"], text=True).strip()
print("HaluEval revision:", DATA_REVISION)


## 3. Normalize response pairs

Each task-specific source item becomes two rows: a reference response (`label=0`) and a hallucinated response (`label=1`). Dialogue history serves as the question. Summarization uses the fixed instruction `Summarize the document faithfully.` as a synthetic question, so its geometry is not directly equivalent to QA. General queries have no context and are retained only in the processed data, not scored with SGI.


In [ ]:
def normalize_text(value):
    return " ".join(str(value).split()) if value is not None else ""


def load_records(path):
    text = path.read_text(encoding="utf-8").strip()
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, list) else [obj]
    except json.JSONDecodeError:
        return [json.loads(line) for line in text.splitlines() if line.strip()]


rows = []
settings = {
    "qa": ("question", "knowledge", "right_answer", "hallucinated_answer"),
    "dialogue": ("dialogue_history", "knowledge", "right_response", "hallucinated_response"),
    "summarization": (None, "document", "right_summary", "hallucinated_summary"),
}
for subset, (q_key, c_key, right_key, hallu_key) in settings.items():
    source = RAW_DIR / "data" / f"{subset}_data.json"
    for i, item in enumerate(load_records(source)):
        q = normalize_text(item.get(q_key)) if q_key else "Summarize the document faithfully."
        c = normalize_text(item.get(c_key))
        pair_id = f"{subset}_{i}"
        for label, key, suffix in [(0, right_key, "right"), (1, hallu_key, "hallu")]:
            r = normalize_text(item.get(key))
            if q and r:
                rows.append(dict(dataset="HaluEval", subset=subset, pair_id=pair_id,
                                 sample_id=f"{pair_id}_{suffix}", question=q, context=c,
                                 response=r, label=label, source_file=str(source)))
source = RAW_DIR / "data" / "general_data.json"
for i, item in enumerate(load_records(source)):
    q, r = normalize_text(item.get("user_query")), normalize_text(item.get("chatgpt_response"))
    value = normalize_text(item.get("hallucination_label", item.get("hallucination"))).lower()
    if q and r:
        rows.append(dict(dataset="HaluEval", subset="general", pair_id=f"general_{i}",
                         sample_id=f"general_{i}", question=q, context="", response=r,
                         label={"yes": 1, "no": 0}.get(value, np.nan), source_file=str(source)))
df = pd.DataFrame(rows)
df.to_parquet(PROCESSED_DIR / "native.parquet", index=False)
df.to_csv(PROCESSED_DIR / "native.csv", index=False)
mask = df[["question", "context", "response"]].ne("").all(axis=1) & df.label.isin([0, 1])
df_ready = df.loc[mask].reset_index(drop=True)
assert len(df_ready) > 0, "No complete labeled triplets found."
display(df.groupby("subset").size().rename("processed_rows").to_frame())
display(df_ready.groupby(["subset", "label"]).size().rename("scored_rows").to_frame())


## 4. Embed and cache complete triplets

Use the original default encoder and preprocessing: `all-mpnet-base-v2`, attention-mask-aware mean pooling, L2 normalization, a 384-token limit and batches of 32. Long contexts are truncated. Cache reuse requires matching input content and configuration; legacy caches without this provenance are recomputed.


In [ ]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
MAX_LENGTH = 384
BATCH_SIZE = 32
CACHE_VERSION = "manual_meanpool_l2_v2"
model_slug = MODEL_NAME.replace("/", "_")
CACHE_DIR = PROJECT_ROOT / "embeddings" / CACHE_VERSION / model_slug / "HaluEval" / "complete_triplets"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fields = ["question", "context", "response"]
fingerprint = hashlib.sha256(df_ready.to_json(orient="records").encode()).hexdigest()
config = dict(model_name=MODEL_NAME, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
              pooling="manual_mean_pooling", normalization="l2", rows=len(df_ready),
              input_sha256=fingerprint, data_revision=DATA_REVISION, environment=versions)
config_path = CACHE_DIR / "config.json"
cache_valid = (config_path.exists() and json.loads(config_path.read_text()) == config
               and all((CACHE_DIR / f"{field}.npy").exists() for field in fields)
               and (CACHE_DIR / "metadata.parquet").exists())


def mean_pooling(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


if not cache_valid:
    # Invalidate the completion marker before overwriting any artifacts.
    config_path.unlink(missing_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    for field in fields:
        parts = []
        texts = df_ready[field].tolist()
        for start in tqdm(range(0, len(texts), BATCH_SIZE), desc=field):
            encoded = tokenizer(texts[start:start+BATCH_SIZE], padding=True,
                                truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
            with torch.inference_mode():
                output = model(**encoded)
                pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
                parts.append(F.normalize(pooled, p=2, dim=1).cpu().numpy().astype(np.float32))
        np.save(CACHE_DIR / f"{field}.npy", np.vstack(parts))
    df_ready.to_parquet(CACHE_DIR / "metadata.parquet", index=False)
    config_path.write_text(json.dumps(config, indent=2))
    del model, tokenizer, parts, encoded, output, pooled
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Validated cache hit:", CACHE_DIR)


## 5. Compute SGI and descriptive evaluation

Evaluation stage. Summary results from the first Colab run are recorded in README.md and results/HaluEval/. Lower SGI is treated as more hallucination-like, so AUROC uses `-SGI` with hallucination as the positive class. Report each subset separately; the pooled result is descriptive. Cohen's d uses `(mean_grounded - mean_hallucinated) / pooled_standard_deviation`.

No classifier or threshold is fitted. These are full-dataset descriptive measurements, not held-out classifier results. A bootstrap or future train/test split must group both responses by `pair_id`. Near-zero response–context angles are excluded because the ratio is unstable.


In [ ]:
from sklearn.metrics import roc_auc_score


def angular_distance(a, b):
    return np.arccos(np.clip(np.sum(a.astype(np.float64) * b.astype(np.float64), axis=1), -1.0, 1.0))


q, c, r = [np.load(CACHE_DIR / f"{field}.npy") for field in fields]
assert q.shape == c.shape == r.shape and q.shape[0] == len(df_ready)
for emb in [q, c, r]:
    assert np.isfinite(emb).all() and np.allclose(np.linalg.norm(emb, axis=1), 1, atol=1e-5)
scored = df_ready.copy()
scored["theta_rq"] = angular_distance(r, q)
scored["theta_rc"] = angular_distance(r, c)
scored["theta_qc"] = angular_distance(q, c)
ANGLE_EPS = 1e-6
valid = scored.theta_rc > ANGLE_EPS
scored["sgi"] = np.nan
scored.loc[valid, "sgi"] = scored.loc[valid, "theta_rq"] / scored.loc[valid, "theta_rc"]
scored.to_parquet(RESULTS_DIR / "sgi_scores.parquet", index=False)


def summarize(name, frame):
    usable = frame.dropna(subset=["sgi"])
    grounded = usable.loc[usable.label == 0, "sgi"].to_numpy()
    hallucinated = usable.loc[usable.label == 1, "sgi"].to_numpy()
    d, auc = np.nan, np.nan
    if len(grounded) > 1 and len(hallucinated) > 1:
        pooled_var = ((len(grounded)-1)*grounded.var(ddof=1)
                      + (len(hallucinated)-1)*hallucinated.var(ddof=1)) / (len(usable)-2)
        if pooled_var > 0:
            d = (grounded.mean()-hallucinated.mean()) / np.sqrt(pooled_var)
    if usable.label.nunique() == 2:
        auc = roc_auc_score(usable.label, -usable.sgi)
    return dict(subset=name, rows=len(frame), excluded_rows=len(frame)-len(usable),
                grounded_rows=len(grounded), hallucinated_rows=len(hallucinated),
                mean_sgi_grounded=grounded.mean() if len(grounded) else np.nan,
                mean_sgi_hallucinated=hallucinated.mean() if len(hallucinated) else np.nan,
                cohens_d=d, auroc=auc)


metrics = pd.DataFrame([summarize(name, frame) for name, frame in scored.groupby("subset")]
                       + [summarize("all", scored)])
metrics.to_csv(RESULTS_DIR / "sgi_metrics.csv", index=False)
(RESULTS_DIR / "run_config.json").write_text(json.dumps(dict(config, angle_epsilon=ANGLE_EPS,
    positive_label=1, auroc_score="-sgi", seed=SEED), indent=2))
display(metrics)


## 6. Compare SGI thresholds

Run this section after mounting Drive. It reads the existing `sgi_scores.parquet`, so it does not need the model, embeddings or a GPU. Set `PROJECT_ROOT` below to match your Drive folder. In a fresh runtime, install `numpy`, `pandas` and `pyarrow` first.

We predict hallucination when `SGI < threshold`, using thresholds 1.00 to 2.00 inclusive in steps of 0.05. Equality is grounded. This is a descriptive comparison on the scored data, not a held-out threshold-selection experiment. No direction is flipped for dialogue or summarization. A grid starting at 1.00 does not explore lower QA thresholds.

The section exports accuracy, precision, recall, F1, confusion counts, accuracy changes from 1.00, and an `ANALYSIS.md` report. Review that report and copy it from Drive into the repository after the run. No new accuracy results have been recorded in this notebook yet.


In [ ]:
# This section can run on its own after setup, using saved scores on Drive.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RESULTS_DIR = PROJECT_ROOT / "results" / "HaluEval"
SCORES_PATH = RESULTS_DIR / "sgi_scores.parquet"

# Integer steps avoid accumulating floating-point error in the threshold grid.
THRESHOLD_START = 1.00
THRESHOLD_STOP = 2.00
THRESHOLD_STEP = 0.05
start_units = int(round(THRESHOLD_START * 100))
stop_units = int(round(THRESHOLD_STOP * 100))
step_units = int(round(THRESHOLD_STEP * 100))
if step_units <= 0 or stop_units < start_units or start_units != 100:
    raise ValueError("Use a positive step, stop >= start, and start=1.00 for the reference comparison.")
THRESHOLDS = np.round(np.arange(start_units, stop_units + 1, step_units) / 100, 2)
TASKS = ["qa", "dialogue", "summarization", "all"]


def threshold_metrics(frame, thresholds):
    required = {"subset", "sgi", "label"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing score columns: {sorted(missing)}")
    scores = pd.to_numeric(frame["sgi"], errors="coerce")
    labels = pd.to_numeric(frame["label"], errors="coerce")
    valid = np.isfinite(scores) & labels.isin([0, 1])
    work = frame.loc[valid, ["subset"]].copy()
    work["sgi"] = scores.loc[valid]
    work["label"] = labels.loc[valid].astype(int)
    result = []
    for task in TASKS:
        original = frame if task == "all" else frame.loc[frame["subset"] == task]
        group = work if task == "all" else work.loc[work["subset"] == task]
        truth = group["label"].to_numpy() == 1
        values = group["sgi"].to_numpy()
        for threshold in thresholds:
            predicted = values < float(threshold)
            tp = int(np.sum(predicted & truth))
            fp = int(np.sum(predicted & ~truth))
            tn = int(np.sum(~predicted & ~truth))
            fn = int(np.sum(~predicted & truth))
            n = len(group)
            result.append(dict(
                subset=task, threshold=float(threshold), rows=len(original),
                evaluated_rows=n, excluded_rows=len(original)-n,
                tp=tp, fp=fp, tn=tn, fn=fn,
                accuracy=(tp+tn)/n if n else np.nan,
                precision=tp/(tp+fp) if tp+fp else np.nan,
                recall=tp/(tp+fn) if tp+fn else np.nan,
                f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan,
                predicted_hallucination_rate=(tp+fp)/n if n else np.nan,
            ))
    return pd.DataFrame(result)


def accuracy_markdown(table):
    lines = ["| SGI threshold | QA accuracy | Dialogue accuracy | Summarization accuracy | All accuracy |",
             "| ---: | ---: | ---: | ---: | ---: |"]
    pivot = table.pivot(index="threshold", columns="subset", values="accuracy")
    for threshold, row in pivot.sort_index().iterrows():
        cells = [f"{100*row[task]:.2f}%" if pd.notna(row[task]) else "NaN" for task in TASKS]
        lines.append(f"| {threshold:.2f} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


if not SCORES_PATH.exists():
    raise FileNotFoundError(
        f"No saved scores at {SCORES_PATH}. Mount Drive and run SGI scoring first."
    )
score_frame = pd.read_parquet(SCORES_PATH)
threshold_table = threshold_metrics(score_frame, THRESHOLDS)
threshold_table.to_csv(RESULTS_DIR / "threshold_metrics.csv", index=False)
comparison = threshold_table.pivot(index="threshold", columns="subset", values="accuracy")[TASKS]
comparison.to_csv(RESULTS_DIR / "threshold_accuracy.csv", float_format="%.8f")

# Changes are relative to the prespecified threshold 1.00, in percentage points.
accuracy_change = comparison.sub(comparison.loc[1.00]).mul(100)
accuracy_change.to_csv(RESULTS_DIR / "threshold_accuracy_change_pp.csv", float_format="%.8f")

sha256 = hashlib.sha256()
with SCORES_PATH.open("rb") as score_file:
    for block in iter(lambda: score_file.read(1024*1024), b""):
        sha256.update(block)
sweep_config = dict(
    created_utc=datetime.now(timezone.utc).isoformat(),
    scores_file=SCORES_PATH.name, scores_sha256=sha256.hexdigest(),
    threshold_start=THRESHOLD_START, threshold_stop=THRESHOLD_STOP,
    threshold_step=THRESHOLD_STEP, thresholds=THRESHOLDS.tolist(),
    prediction_rule="hallucination if sgi < threshold; equality is grounded",
    positive_label=1, evaluation="descriptive sweep on saved labeled scores; no test split",
    nonfinite_scores="excluded", undefined_ratios="NaN",
)
(RESULTS_DIR / "threshold_sweep_config.json").write_text(json.dumps(sweep_config, indent=2))

report = """# Experiment analysis

## Scope

This project tests SGI on HaluEval. It follows the paper's geometric idea but does not fully replicate its experiment. QA is the comparison task. Dialogue uses conversation history as q; summarization uses our fixed instruction as q. Causes of the task differences remain untested.

## Recorded baseline

The November 4 run used all-mpnet-base-v2, a 384-token limit and manual masked mean pooling with L2 normalization. Its AUROC values were QA 0.73986109, dialogue 0.47446514, summarization 0.40841045 and pooled 0.53972089. These historical values are not recomputed by this threshold sweep.

## Approach 1: threshold sensitivity

Rule: predict hallucination when SGI < threshold. Equality is classified as grounded. The fixed grid is 1.00 through 2.00, inclusive, in steps of 0.05. This keeps the same scoring direction for every task. AUROC is independent of this chosen grid; accuracy, precision, recall, F1 and confusion counts depend on the threshold.

The table below reports descriptive accuracy on the saved scores. It is not an independent test of a selected threshold. Inspecting all rows and reporting the highest value as test performance would overstate the result. The current grid does not cover thresholds below 1.00, where QA may also have useful operating points.

<!-- THRESHOLD_TABLE_START -->
""" + accuracy_markdown(threshold_table) + """
<!-- THRESHOLD_TABLE_END -->

Accuracy is the fraction of labels predicted correctly. CSV values are fractions; this table shows percentages. The sweep exports precision, recall, F1 and TP/FP/TN/FN for every task and threshold. Undefined ratios are NaN. Nonfinite SGI values and invalid labels are excluded and counted.

## Conclusions and next checks

The baseline suggests useful QA ranking, weak dialogue ranking and reversed summarization ranking. It does not establish why. The sweep measures how decisions change as the cutoff increases; it does not change or improve AUROC. Increasing the threshold can only increase or preserve the number of hallucination predictions. Recall cannot decrease, while false alarms cannot decrease; accuracy and precision can move in either direction.

An always-one-class prediction would score 50% on the balanced November 4 task subsets. Compare threshold accuracy with that baseline. The pooled result does not replace task-specific conclusions. No threshold has been selected or validated for deployment.

- [ ] Inspect score distributions and small response-context angles.
- [ ] Compare SGI against response-context similarity.
- [ ] Test alternative summarization inputs using a development protocol.
- [ ] Select thresholds on development pairs and evaluate once on separate test pairs.
- [ ] Add bootstrap intervals grouped by pair_id.

## Run evidence

Source score hash and grid settings are saved in threshold_sweep_config.json. Detailed counts are in threshold_metrics.csv; accuracy is in threshold_accuracy.csv and changes from threshold 1.00 are in threshold_accuracy_change_pp.csv. Preserve these files with the run before changing settings. Replace the repository's ANALYSIS.md with this report after reviewing the Colab output.
"""
(RESULTS_DIR / "ANALYSIS.md").write_text(report)
print("Accuracy by threshold (fractions):")
display(comparison)
print("Accuracy change from threshold 1.00 (percentage points):")
display(accuracy_change)
display(Markdown(accuracy_markdown(threshold_table)))
print("Saved threshold metrics and ANALYSIS.md to:", RESULTS_DIR)


## Interpretation

Compare the QA row with the paper's SGI results; dialogue and summarization are additional task-specific measurements. An AUROC near 0.5 indicates weak ranking ability. SGI > 1 only means the response is angularly closer to context than to question; it is not a calibrated truth threshold. Report preprocessing, truncation, exclusions and model choice alongside any metrics. See README.md for the recorded project status and research reference.
